In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import sys
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

print("Python:", sys.version.split()[0])

for ten_thu_vien in ["pandas", "torch", "transformers", "tokenizers"]:
    try:
        print(ten_thu_vien, ":", version(ten_thu_vien))
    except PackageNotFoundError:
        print(ten_thu_vien, ": chưa cài")

thu_muc_du_lieu = Path("/content/drive/MyDrive/CMU-MOSI")

duong_dan_manifest = (
    thu_muc_du_lieu / "preprocessing" / "shared" / "manifest.csv"
)

duong_dan_pilot = (
    thu_muc_du_lieu / "preprocessing" / "shared" / "pilot_manifest.csv"
)

for duong_dan in [duong_dan_manifest, duong_dan_pilot]:
    print(duong_dan)
    print("Tồn tại:", duong_dan.is_file())

Python: 3.13.18
pandas : 2.2.3
torch : 2.11.0+cpu
transformers : 5.21.0
tokenizers : 0.23.3
/content/drive/MyDrive/CMU-MOSI/preprocessing/shared/manifest.csv
Tồn tại: True
/content/drive/MyDrive/CMU-MOSI/preprocessing/shared/pilot_manifest.csv
Tồn tại: True


In [ ]:
import pandas as pd

kieu_cot = {
    "id": "string",
    "video_id": "string",
    "clip_id": "string",
}

manifest = pd.read_csv(
    duong_dan_manifest,
    dtype=kieu_cot,
    keep_default_na=False,
)

pilot = pd.read_csv(
    duong_dan_pilot,
    dtype=kieu_cot,
    keep_default_na=False,
)

assert len(manifest) == 2199, "Số mẫu manifest chưa đúng"
assert manifest["id"].is_unique, "Manifest có ID trùng"
assert pilot["id"].is_unique, "Pilot có ID trùng"

id_tao_lai = manifest["video_id"] + "$_$" + manifest["clip_id"]
assert manifest["id"].eq(id_tao_lai).all(), "Có ID sai quy tắc"

assert pilot["id"].isin(manifest["id"]).all(), "Pilot có ID ngoài manifest"

so_mau_theo_split = manifest["split"].value_counts().to_dict()
assert so_mau_theo_split == {
    "train": 1284,
    "valid": 229,
    "test": 686,
}, "Số mẫu từng split chưa đúng"

assert manifest["raw_text"].str.strip().ne("").all(), "Có transcript trống"

print("Số mẫu manifest:", len(manifest))
print("Số mẫu pilot:", len(pilot))
print("Phân bố split:", so_mau_theo_split)

mau_thu = pilot.iloc[0]

print("\nID thử:", mau_thu["id"])
print("Transcript gốc:", repr(mau_thu["raw_text"]))
print("Nhãn:", mau_thu["label"])

Số mẫu manifest: 2199
Số mẫu pilot: 10
Phân bố split: {'train': 1284, 'test': 686, 'valid': 229}

ID thử: 7JsX8y1ysxY$_$12
Transcript gốc: 'I DONT KNOW CLASSIC SUPERHERO KIND OF GETTING ON KIND OF THING'
Nhãn: 0.6


In [ ]:
import json
import hashlib
import numpy as np
from IPython.display import Audio, display

# Sửa đường dẫn này nếu chọn bộ kết quả Text khác.
tep_text = thu_muc_du_lieu / "preprocessing/text/pilot/20261101T105527_853034Z/text_pilot_results.json"
assert tep_text.is_file(), "Không tìm thấy JSON Text đã kiểm chứng"
ket_qua_pilot = json.loads(tep_text.read_text(encoding="utf-8"))
assert set(ket_qua_pilot) == set(pilot["id"].astype(str))


In [ ]:
# phần này dùng transcript và WAV để chạy MFA, lấy thời gian từng từ.

from pathlib import Path

thu_muc_wav = Path(
    "/content/drive/MyDrive/CMU-MOSI/preprocessing/audio/wav"
)

id_thu = "7JsX8y1ysxY$_$12"
duong_dan_wav = thu_muc_wav / f"{id_thu}.wav"

print("Thư mục WAV tồn tại:", thu_muc_wav.is_dir())
print("WAV thử tồn tại:", duong_dan_wav.is_file())
print("Đường dẫn:", duong_dan_wav)

Thư mục WAV tồn tại: True
WAV thử tồn tại: True
Đường dẫn: /content/drive/MyDrive/CMU-MOSI/preprocessing/audio/wav/7JsX8y1ysxY$_$12.wav


In [ ]:
import wave

with wave.open(str(duong_dan_wav), "rb") as tep_wav:
    so_kenh = tep_wav.getnchannels()
    tan_so = tep_wav.getframerate()
    so_byte_moi_mau = tep_wav.getsampwidth()
    so_mau = tep_wav.getnframes()
    kieu_nen = tep_wav.getcomptype()

thoi_luong = so_mau / tan_so

print("ID:", id_thu)
print("Số kênh:", so_kenh)
print("Tần số lấy mẫu:", tan_so, "Hz")
print("Bit depth:", so_byte_moi_mau * 8)
print("Thời lượng:", thoi_luong, "giây")
print("Kiểu nén:", kieu_nen)

assert so_kenh == 1, "WAV chưa phải mono"
assert tan_so == 16000, "WAV chưa đúng 16 kHz"
assert so_byte_moi_mau == 2, "WAV chưa đúng 16-bit"
assert kieu_nen == "NONE", "WAV phải là PCM không nén"
assert so_mau > 0, "WAV không có mẫu âm thanh"

print("WAV đạt các kiểm tra định dạng.")

ID: 7JsX8y1ysxY$_$12
Số kênh: 1
Tần số lấy mẫu: 16000 Hz
Bit depth: 16
Thời lượng: 5.2346875 giây
Kiểu nén: NONE
WAV đạt các kiểm tra định dạng.


In [ ]:
from pathlib import Path
import pandas as pd
import zipfile
import shutil

thu_muc_du_lieu = Path("/content/drive/MyDrive/CMU-MOSI")

duong_dan_manifest = (
    thu_muc_du_lieu
    / "preprocessing"
    / "shared"
    / "manifest.csv"
)

assert duong_dan_manifest.is_file(), "Không tìm thấy manifest.csv"

manifest = pd.read_csv(
    duong_dan_manifest,
    dtype={
        "id": "string",
        "video_id": "string",
        "clip_id": "string",
    },
    keep_default_na=False,
)

print("Đã cấu hình thư mục:", thu_muc_du_lieu)
print("Đã đọc manifest:", len(manifest), "mẫu")
print("Raw.zip tồn tại:", (thu_muc_du_lieu / "Raw.zip").is_file())

Đã cấu hình thư mục: /content/drive/MyDrive/CMU-MOSI
Đã đọc manifest: 2199 mẫu
Raw.zip tồn tại: True


In [ ]:
import zipfile
import shutil

# Dùng đường dẫn đã cấu hình trong notebook
duong_dan_zip = thu_muc_du_lieu / "Raw.zip"

dong_thu = manifest.loc[manifest["id"] == id_thu]
assert len(dong_thu) == 1, "Không tìm thấy duy nhất một mẫu theo ID"

video_relpath = dong_thu.iloc[0]["video_relpath"]

# Video dùng kiểm tra tạm trong phiên Colab
thu_muc_kiem_tra = Path("/content/mosi_alignment_check")
thu_muc_kiem_tra.mkdir(parents=True, exist_ok=True)

duong_dan_video = thu_muc_kiem_tra / f"{id_thu}.mp4"

assert duong_dan_zip.is_file(), "Không tìm thấy Raw.zip"

with zipfile.ZipFile(duong_dan_zip, "r") as tep_zip:
    with tep_zip.open(video_relpath) as nguon:
        with duong_dan_video.open("wb") as dich:
            shutil.copyfileobj(nguon, dich)

print("Video trong ZIP:", video_relpath)
print("Đã lấy video tại:", duong_dan_video)
print("Kích thước:", duong_dan_video.stat().st_size, "bytes")

Video trong ZIP: Raw/7JsX8y1ysxY/12.mp4
Đã lấy video tại: /content/mosi_alignment_check/7JsX8y1ysxY$_$12.mp4
Kích thước: 251905 bytes


In [ ]:
import subprocess
import json
import shutil

assert shutil.which("ffprobe") is not None, (
    "Môi trường chưa có ffprobe"
)


def doc_thong_tin_media(duong_dan):
    lenh = [
        "ffprobe",
        "-v", "error",
        "-show_entries",
        "stream=index,codec_type,codec_name,start_time,duration"
        ":format=start_time,duration",
        "-of", "json",
        str(duong_dan),
    ]

    ket_qua = subprocess.run(
        lenh,
        capture_output=True,
        text=True,
    )

    if ket_qua.returncode != 0:
        raise RuntimeError(ket_qua.stderr)

    return json.loads(ket_qua.stdout)


thong_tin_video = doc_thong_tin_media(duong_dan_video)
thong_tin_wav = doc_thong_tin_media(duong_dan_wav)

print("THÔNG TIN VIDEO GỐC")
print(json.dumps(thong_tin_video, indent=2, ensure_ascii=False))

print("\nTHÔNG TIN WAV")
print(json.dumps(thong_tin_wav, indent=2, ensure_ascii=False))

THÔNG TIN VIDEO GỐC
{
  "programs": [],
  "streams": [
    {
      "index": 0,
      "codec_name": "h264",
      "codec_type": "video",
      "start_time": "0.000000",
      "duration": "5.266667"
    },
    {
      "index": 1,
      "codec_name": "aac",
      "codec_type": "audio",
      "start_time": "0.000000",
      "duration": "5.218005"
    }
  ],
  "format": {
    "start_time": "0.000000",
    "duration": "5.266667"
  }
}

THÔNG TIN WAV
{
  "programs": [],
  "streams": [
    {
      "index": 0,
      "codec_name": "pcm_s16le",
      "codec_type": "audio",
      "duration": "5.234687"
    }
  ],
  "format": {
    "duration": "5.234688"
  }
}


In [ ]:
from pathlib import Path
import shutil
import hashlib

# Lấy đúng transcript theo ID
dong_thu = manifest.loc[manifest["id"] == id_thu]
assert len(dong_thu) == 1, "Không tìm thấy duy nhất một mẫu"

raw_text_mfa = dong_thu.iloc[0]["raw_text"]

# Chỉ chuẩn hóa khoảng trắng, không sửa từ
text_mfa = " ".join(raw_text_mfa.split())
assert text_mfa, "Transcript trống"

# Lưu đầu vào trên Drive để giữ lại sau khi ngắt Colab
thu_muc_mfa = (
    thu_muc_du_lieu / "preprocessing" / "alignment" / "one_clip"
)

thu_muc_corpus = thu_muc_mfa / "corpus"
thu_muc_nguoi_noi = (
    thu_muc_corpus / str(dong_thu.iloc[0]["video_id"])
)

thu_muc_nguoi_noi.mkdir(parents=True, exist_ok=True)

wav_mfa = thu_muc_nguoi_noi / f"{id_thu}.wav"
lab_mfa = thu_muc_nguoi_noi / f"{id_thu}.lab"

# Sao chép WAV nguyên bản
if wav_mfa.exists():
    assert wav_mfa.read_bytes() == duong_dan_wav.read_bytes(), (
        "Đã có WAV khác nội dung trong thư mục MFA"
    )
else:
    shutil.copyfile(duong_dan_wav, wav_mfa)

# Không ghi đè transcript khác nội dung
noi_dung_lab = text_mfa + "\n"

if lab_mfa.exists():
    assert lab_mfa.read_text(encoding="utf-8") == noi_dung_lab, (
        "Đã có transcript khác nội dung"
    )
else:
    lab_mfa.write_text(noi_dung_lab, encoding="utf-8")

# Xác nhận WAV sao chép không thay đổi
assert (
    hashlib.sha256(wav_mfa.read_bytes()).hexdigest()
    == hashlib.sha256(duong_dan_wav.read_bytes()).hexdigest()
)

print("Corpus:", thu_muc_corpus)
print("WAV:", wav_mfa.name)
print("Transcript:", lab_mfa.name)
print("Nội dung:", lab_mfa.read_text(encoding="utf-8"))
print("WAV sao chép khớp nguyên bản.")

# Kiểm tra công cụ hiện có trước khi hướng dẫn cài MFA
print("mfa:", shutil.which("mfa"))
print("conda:", shutil.which("conda"))
print("micromamba:", shutil.which("micromamba"))

Corpus: /content/drive/MyDrive/CMU-MOSI/preprocessing/alignment/one_clip/corpus
WAV: 7JsX8y1ysxY$_$12.wav
Transcript: 7JsX8y1ysxY$_$12.lab
Nội dung: I DONT KNOW CLASSIC SUPERHERO KIND OF GETTING ON KIND OF THING

WAV sao chép khớp nguyên bản.
mfa: None
conda: None
micromamba: None


In [ ]:
import subprocess
from pathlib import Path
import platform

assert platform.system() == "Linux"
assert platform.machine() == "x86_64", "Cần kiểm tra lại kiến trúc máy"

thu_muc_cong_cu = Path("/content/mfa_tools")
thu_muc_cong_cu.mkdir(parents=True, exist_ok=True)

tep_cai = thu_muc_cong_cu / "micromamba.tar.bz2"
micromamba = thu_muc_cong_cu / "bin" / "micromamba"

if not micromamba.is_file():
    subprocess.run(
        [
            "curl", "-fL",
            "https://micro.mamba.pm/api/micromamba/linux-64/latest",
            "-o", str(tep_cai),
        ],
        check=True,
    )

    subprocess.run(
        [
            "tar", "-xjf", str(tep_cai),
            "-C", str(thu_muc_cong_cu),
            "bin/micromamba",
        ],
        check=True,
    )

subprocess.run(
    [str(micromamba), "--version"],
    check=True,
)

CompletedProcess(args=['/content/mfa_tools/bin/micromamba', '--version'], returncode=0)

In [ ]:
mfa_env = Path("/content/mfa_env")

if not (mfa_env / "bin" / "mfa").is_file():
    subprocess.run(
        [
            str(micromamba),
            "create",
            "-y",
            "-p", str(mfa_env),
            "-c", "conda-forge",
            "--strict-channel-priority",
            "montreal-forced-aligner",
        ],
        check=True,
    )

# Dùng tiền tố này cho các lệnh MFA sau
lenh_mfa = [
    str(micromamba),
    "run",
    "-p", str(mfa_env),
    "mfa",
]

# Kiểm tra phiên bản bằng Python trong môi trường MFA
phien_ban_mfa = subprocess.run(
    [
        str(micromamba),
        "run",
        "-p", str(mfa_env),
        "python",
        "-c",
        "from importlib.metadata import version; "
        "print(version('montreal-forced-aligner'))",
    ],
    capture_output=True,
    text=True,
)

print("Phiên bản MFA:", phien_ban_mfa.stdout.strip())

if phien_ban_mfa.stderr:
    print(phien_ban_mfa.stderr)

phien_ban_mfa.check_returncode()

# Kiểm tra giao diện dòng lệnh MFA
kiem_tra_mfa = subprocess.run(
    lenh_mfa + ["--help"],
    capture_output=True,
    text=True,
)

print(kiem_tra_mfa.stdout)

if kiem_tra_mfa.stderr:
    print(kiem_tra_mfa.stderr)

kiem_tra_mfa.check_returncode()

Phiên bản MFA: 3.4.3
                                                                                
 Usage: mfa [OPTIONS] COMMAND [ARGS]...                                         
                                                                                
 Montreal Forced Aligner is a command line utility for aligning speech and      
 text.                                                                          
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --help  Show this message and exit.                                          │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─ Commands ───────────────────────────────────────────────────────────────────╮
│ adapt                   Adapt an acoustic model                              │
│ adapt_hf                Adapt an acoustic model                              │
│ align

In [ ]:
for loai in ["acoustic", "dictionary"]:
    print("Đang tải:", loai, "english_us_arpa", flush=True)

    ket_qua_tai = subprocess.run(
        lenh_mfa + [
            "model", "download",
            loai,
            "english_us_arpa",
        ],
        capture_output=True,
        text=True,
    )

    print(ket_qua_tai.stdout)

    if ket_qua_tai.stderr:
        print(ket_qua_tai.stderr)

    ket_qua_tai.check_returncode()

print("Đã hoàn tất tải model và từ điển.")

Đang tải: acoustic english_us_arpa

Đang tải: dictionary english_us_arpa

Đã hoàn tất tải model và từ điển.


In [ ]:
from datetime import datetime, timezone

ma_kiem_tra = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

thu_muc_oov = (
    thu_muc_mfa / f"oov_check_{ma_kiem_tra}"
)

ket_qua_oov = subprocess.run(
    lenh_mfa + [
        "find_oovs",
        str(thu_muc_corpus),
        "english_us_arpa",
        str(thu_muc_oov),
    ],
    capture_output=True,
    text=True,
)

# Lưu thông báo để xem lại cả khi lệnh thất bại
thu_muc_oov.mkdir(parents=True, exist_ok=True)

(thu_muc_oov / "command_output.log").write_text(
    ket_qua_oov.stdout + "\n" + ket_qua_oov.stderr,
    encoding="utf-8",
)

print(ket_qua_oov.stdout)
print(ket_qua_oov.stderr)

ket_qua_oov.check_returncode()

print("\nCác file báo cáo:")

for tep in sorted(thu_muc_oov.rglob("*")):
    if tep.is_file():
        print(tep.relative_to(thu_muc_oov))

        if tep.suffix in {".txt", ".csv"}:
            print(tep.read_text(encoding="utf-8"))

   1%                                      1/100  [ 0:00:01 < -:--:-- , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:01 < 0:00:00 , ? it/s ]

 INFO     Setting up corpus information...                                      
 INFO     Loading corpus from source files...                                   
 INFO     Found 1 speaker across 1 file, average number of utterances per       
          speaker: 1.0                                                          
 INFO     Initializing multiprocessing jobs...                                  
 WARNING  Number of jobs was specified as 3, but due to only having 1 speakers, 
          MFA will only use

In [ ]:
tai_g2p = subprocess.run(
    lenh_mfa + [
        "model", "download",
        "g2p",
        "english_us_arpa",
    ],
    capture_output=True,
    text=True,
)

print(tai_g2p.stdout)
print(tai_g2p.stderr)

tai_g2p.check_returncode()

In [ ]:
from datetime import datetime, timezone

# Dùng danh sách từ thiếu do MFA đã xuất
tep_tu_thieu = (
    thu_muc_oov / "oovs_found_english_us_arpa.txt"
)

assert tep_tu_thieu.is_file(), "Không tìm thấy danh sách OOV"

danh_sach_oov = [
    dong.strip()
    for dong in tep_tu_thieu.read_text(encoding="utf-8").splitlines()
    if dong.strip()
]

print("Các từ cần bổ sung:", danh_sach_oov)

assert danh_sach_oov, "Danh sách OOV trống"

ma_g2p = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

thu_muc_g2p = thu_muc_mfa / f"g2p_{ma_g2p}"
thu_muc_g2p.mkdir(parents=True, exist_ok=False)

tep_phat_am = thu_muc_g2p / "oov_pronunciations.dict"

ket_qua_g2p = subprocess.run(
    lenh_mfa + [
        "g2p",
        str(tep_tu_thieu),
        "english_us_arpa",
        str(tep_phat_am),
    ],
    capture_output=True,
    text=True,
)

(thu_muc_g2p / "command_output.log").write_text(
    ket_qua_g2p.stdout + "\n" + ket_qua_g2p.stderr,
    encoding="utf-8",
)

print(ket_qua_g2p.stdout)
print(ket_qua_g2p.stderr)

ket_qua_g2p.check_returncode()

assert tep_phat_am.is_file(), "G2P chưa tạo file phát âm"

print("\nCách phát âm được đề xuất:")
print(tep_phat_am.read_text(encoding="utf-8"))

Các từ cần bổ sung: ['superhero']
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:01 < 0:00:00 , ? it/s ]

 INFO     Generating pronunciations...                                          
 INFO     Done! Everything took 3.421 seconds                                   


Cách phát âm được đề xuất:
superhero	S UW2 P ER0 HH IH1 R OW0
superhero	S UW2 P ER0 HH IY1 R OW0



In [ ]:
# Chạy Python trong môi trường MFA để tìm từ điển đã tải
tim_tu_dien = subprocess.run(
    [
        str(micromamba),
        "run", "-p", str(mfa_env),
        "python", "-c",
        "from montreal_forced_aligner.models import DictionaryModel; "
        "print(DictionaryModel.get_pretrained_path('english_us_arpa'))",
    ],
    capture_output=True,
    text=True,
)

if tim_tu_dien.returncode != 0:
    raise RuntimeError(tim_tu_dien.stderr)

tu_dien_goc = Path(tim_tu_dien.stdout.strip())

assert tu_dien_goc.is_file(), (
    f"Không tìm thấy từ điển: {tu_dien_goc}"
)

# Lưu bản riêng trong thư mục G2P của lần chạy này
tu_dien_bo_sung = thu_muc_g2p / "english_us_arpa_custom.dict"

# Chế độ xb tránh ghi đè file đã tồn tại
with tu_dien_goc.open("rb") as nguon:
    with tu_dien_bo_sung.open("xb") as dich:
        shutil.copyfileobj(nguon, dich)

print("Từ điển gốc:", tu_dien_goc)
print("Bản riêng:", tu_dien_bo_sung)

Từ điển gốc: /root/Documents/MFA/pretrained_models/dictionary/english_us_arpa.dict
Bản riêng: /content/drive/MyDrive/CMU-MOSI/preprocessing/alignment/one_clip/g2p_20261102T093251_320219Z/english_us_arpa_custom.dict


In [ ]:
them_phat_am = subprocess.run(
    lenh_mfa + [
        "model", "add_words",
        str(tu_dien_bo_sung),
        str(tep_phat_am),
    ],
    capture_output=True,
    text=True,
)

print(them_phat_am.stdout)
print(them_phat_am.stderr)

them_phat_am.check_returncode()

# Đọc lại để xác nhận có mục superhero
cac_dong_superhero = []

with tu_dien_bo_sung.open("r", encoding="utf-8") as tep:
    for dong in tep:
        thanh_phan = dong.split()

        if thanh_phan and thanh_phan[0].lower() == "superhero":
            cac_dong_superhero.append(dong.strip())

assert cac_dong_superhero, "Chưa tìm thấy superhero trong từ điển riêng"

print("\nCác mục superhero trong từ điển riêng:")

for dong in cac_dong_superhero:
    print(dong)




Các mục superhero trong từ điển riêng:
superhero	S UW2 P ER0 HH IH1 R OW0
superhero	S UW2 P ER0 HH IY1 R OW0


In [ ]:
from datetime import datetime, timezone

ma_kiem_tra_lai = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

thu_muc_oov_moi = (
    thu_muc_mfa / f"oov_custom_{ma_kiem_tra_lai}"
)

# Dùng thư mục tạm riêng để không tái sử dụng kết quả cũ
thu_muc_tam_oov = Path(
    f"/content/mfa_oov_custom_{ma_kiem_tra_lai}"
)

kiem_tra_oov_moi = subprocess.run(
    lenh_mfa + [
        "find_oovs",
        str(thu_muc_corpus),
        str(tu_dien_bo_sung),
        str(thu_muc_oov_moi),
        "--temporary_directory", str(thu_muc_tam_oov),
        "--num_jobs", "1",
    ],
    capture_output=True,
    text=True,
)

thu_muc_oov_moi.mkdir(parents=True, exist_ok=True)

(thu_muc_oov_moi / "command_output.log").write_text(
    kiem_tra_oov_moi.stdout + "\n" + kiem_tra_oov_moi.stderr,
    encoding="utf-8",
)

print(kiem_tra_oov_moi.stdout)
print(kiem_tra_oov_moi.stderr)

kiem_tra_oov_moi.check_returncode()

print("\nNỘI DUNG CÁC BÁO CÁO")

for tep in sorted(thu_muc_oov_moi.rglob("*")):
    if tep.is_file() and tep.suffix in {".txt", ".csv"}:
        print("\nFile:", tep.name)

        noi_dung = tep.read_text(encoding="utf-8")

        print(noi_dung if noi_dung.strip() else "(file trống)")

   1%                                      1/100  [ 0:00:01 < -:--:-- , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:01 < 0:00:00 , ? it/s ]

 INFO     Setting up corpus information...                                      
 INFO     Loading corpus from source files...                                   
 INFO     Found 1 speaker across 1 file, average number of utterances per       
          speaker: 1.0                                                          
 INFO     Initializing multiprocessing jobs...                                  
 INFO     Normalizing text...                                                   
 INFO     Generating MFCCs.

In [ ]:
from datetime import datetime, timezone

ma_alignment = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

thu_muc_lan_alignment = (
    thu_muc_mfa / f"alignment_{ma_alignment}"
)

thu_muc_lan_alignment.mkdir(parents=True, exist_ok=False)

thu_muc_textgrid = thu_muc_lan_alignment / "textgrids"

thu_muc_tam_alignment = Path(
    f"/content/mfa_alignment_{ma_alignment}"
)

lenh_alignment = lenh_mfa + [
    "align",
    str(thu_muc_corpus),
    str(tu_dien_bo_sung),
    "english_us_arpa",
    str(thu_muc_textgrid),
    "--temporary_directory", str(thu_muc_tam_alignment),
    "--num_jobs", "1",
]

# Lưu cấu hình của lần chạy
cau_hinh_alignment = {
    "id": id_thu,
    "mfa_version": "3.4.3",
    "acoustic_model": "english_us_arpa",
    "dictionary_path": str(tu_dien_bo_sung),
    "dictionary_sha256": hashlib.sha256(
        tu_dien_bo_sung.read_bytes()
    ).hexdigest(),
    "corpus_directory": str(thu_muc_corpus),
    "command": lenh_alignment,
}

(thu_muc_lan_alignment / "config.json").write_text(
    json.dumps(
        cau_hinh_alignment,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("Đang chạy alignment...", flush=True)

ket_qua_alignment = subprocess.run(
    lenh_alignment,
    capture_output=True,
    text=True,
)

# Lưu log trước khi kiểm tra lỗi
(thu_muc_lan_alignment / "command_output.log").write_text(
    ket_qua_alignment.stdout + "\n" + ket_qua_alignment.stderr,
    encoding="utf-8",
)

print(ket_qua_alignment.stdout)
print(ket_qua_alignment.stderr)

ket_qua_alignment.check_returncode()

cac_tep_textgrid = sorted(
    thu_muc_textgrid.rglob("*.TextGrid")
)

assert len(cac_tep_textgrid) == 1, (
    f"Mong đợi 1 TextGrid, thực tế có {len(cac_tep_textgrid)}"
)

tep_textgrid = cac_tep_textgrid[0]

assert tep_textgrid.stem == id_thu, "TextGrid không đúng ID"
assert tep_textgrid.stat().st_size > 0, "TextGrid rỗng"

print("\nĐã tạo TextGrid:", tep_textgrid)
print("Kích thước:", tep_textgrid.stat().st_size, "bytes")
print("Thư mục lưu kết quả:", thu_muc_lan_alignment)

Đang chạy alignment...
   1%                                      1/100  [ 0:00:01 < -:--:-- , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:03 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:00 < 0:00:00 , ? it/s ]

 INFO     Setting up corpus information...                                      
 INF

In [ ]:
# Đọc bằng Python của môi trường MFA
code_doc_textgrid = """
import json
import sys
from praatio import textgrid

tg = textgrid.openTextgrid(
    sys.argv[1],
    includeEmptyIntervals=True,
)

ket_qua = {}

for ten_tier in tg.tierNames:
    tier = tg.getTier(ten_tier)
    ket_qua[ten_tier] = [list(muc) for muc in tier.entries]

print(json.dumps(ket_qua, ensure_ascii=False))
"""

doc_textgrid = subprocess.run(
    [
        str(micromamba),
        "run", "-p", str(mfa_env),
        "python", "-c", code_doc_textgrid,
        str(tep_textgrid),
    ],
    capture_output=True,
    text=True,
)

if doc_textgrid.returncode != 0:
    raise RuntimeError(doc_textgrid.stderr)

du_lieu_textgrid = json.loads(doc_textgrid.stdout)

print("Các tier:", list(du_lieu_textgrid))

# Chỉ chọn khi xác định được duy nhất một tier từ
cac_tier_tu = [
    ten for ten in du_lieu_textgrid
    if ten.lower() == "words" or ten.lower().endswith(" - words")
]

assert len(cac_tier_tu) == 1, (
    "Chưa xác định được duy nhất tier words. "
    f"Các tier hiện có: {list(du_lieu_textgrid)}"
)

ten_tier_tu = cac_tier_tu[0]

bang_khoang = pd.DataFrame(
    du_lieu_textgrid[ten_tier_tu],
    columns=["start", "end", "word"],
)

bang_khoang["duration"] = (
    bang_khoang["end"] - bang_khoang["start"]
)

display(bang_khoang)

# Tách các khoảng có nhãn từ để đối chiếu,
# vẫn giữ bảng đầy đủ trong bang_khoang
bang_tu_mfa = bang_khoang.loc[
    bang_khoang["word"].str.strip().ne("")
].copy().reset_index(drop=True)

print("Số khoảng có nhãn từ:", len(bang_tu_mfa))

Các tier: ['words', 'phones']


,start,end,word,duration
0,0.00,0.060000,,0.060000
1,0.06,0.090000,i,0.030000
2,0.09,0.780000,dont,0.690000
3,0.78,0.840000,know,0.060000
4,0.84,1.190000,classic,0.350000
5,1.19,1.330000,,0.140000
6,1.33,1.990000,superhero,0.660000
7,1.99,2.260000,kind,0.270000
8,2.26,2.360000,of,0.100000
9,2.36,2.910000,,0.550000


Số khoảng có nhãn từ: 12


In [ ]:
import math
import wave
from itertools import zip_longest

words_goc = " ".join(
    manifest.loc[manifest["id"] == id_thu, "raw_text"].iloc[0].split()
).split()

words_mfa = bang_tu_mfa["word"].tolist()

bang_doi_chieu = pd.DataFrame(
    list(zip_longest(words_goc, words_mfa, fillvalue=None)),
    columns=["word_goc", "word_mfa"],
)

bang_doi_chieu["khop_bo_qua_hoa_thuong"] = [
    goc is not None
    and mfa is not None
    and goc.casefold() == mfa.casefold()
    for goc, mfa in zip_longest(words_goc, words_mfa, fillvalue=None)
]

display(bang_doi_chieu)

# Đọc lại thời lượng WAV, tránh phụ thuộc biến cũ
with wave.open(str(wav_mfa), "rb") as tep:
    thoi_luong_wav = tep.getnframes() / tep.getframerate()

loi_thoi_gian = []
ket_thuc_truoc = None

for vi_tri, dong in bang_khoang.iterrows():
    bat_dau = float(dong["start"])
    ket_thuc = float(dong["end"])

    if not (math.isfinite(bat_dau) and math.isfinite(ket_thuc)):
        loi_thoi_gian.append(f"Khoảng {vi_tri}: timestamp không hữu hạn")
        continue

    if not (0 <= bat_dau < ket_thuc <= thoi_luong_wav + 0.001):
        loi_thoi_gian.append(f"Khoảng {vi_tri}: biên thời gian không hợp lệ")

    if ket_thuc_truoc is not None and bat_dau < ket_thuc_truoc - 0.000001:
        loi_thoi_gian.append(f"Khoảng {vi_tri}: chồng lấn khoảng trước")

    ket_thuc_truoc = ket_thuc

khop_danh_sach_tu = bool(
    bang_doi_chieu["khop_bo_qua_hoa_thuong"].all()
)

print("Số từ gốc:", len(words_goc))
print("Số từ MFA:", len(words_mfa))
print("Khớp toàn bộ từ, bỏ qua hoa/thường:", khop_danh_sach_tu)
print("Lỗi cấu trúc thời gian:", loi_thoi_gian)

if not khop_danh_sach_tu:
    print("Cần kiểm tra khác biệt từ; chưa ghép timestamp vào words.")

,word_goc,word_mfa,khop_bo_qua_hoa_thuong
0,I,i,True
1,DONT,dont,True
2,KNOW,know,True
3,CLASSIC,classic,True
4,SUPERHERO,superhero,True
5,KIND,kind,True
6,OF,of,True
7,GETTING,getting,True
8,ON,on,True
9,KIND,kind,True


Số từ gốc: 12
Số từ MFA: 12
Khớp toàn bộ từ, bỏ qua hoa/thường: True
Lỗi cấu trúc thời gian: []


In [ ]:
import wave
import numpy as np
from IPython.display import Audio, display

# Đọc WAV, không chỉnh sửa file
with wave.open(str(wav_mfa), "rb") as tep:
    tan_so_nghe = tep.getframerate()

    am_thanh = np.frombuffer(
        tep.readframes(tep.getnframes()),
        dtype="<i2",
    ).astype(np.float32) / 32768.0

assert khop_danh_sach_tu
assert not loi_thoi_gian

print("TOÀN BỘ CLIP")
print("Transcript:", " ".join(words_goc))

display(Audio(
    am_thanh,
    rate=tan_so_nghe,
    normalize=False,
))

# Chỉ số theo danh sách 12 từ, bắt đầu từ 0
cac_doan_kiem_tra = [
    ("I DONT KNOW", 0, 2),
    ("CLASSIC SUPERHERO KIND OF", 3, 6),
    ("ON KIND OF THING", 8, 11),
]

for ten_doan, tu_dau, tu_cuoi in cac_doan_kiem_tra:
    bat_dau = float(bang_tu_mfa.iloc[tu_dau]["start"])
    ket_thuc = float(bang_tu_mfa.iloc[tu_cuoi]["end"])

    # Thêm ngữ cảnh 0,1 giây ở hai bên để dễ nghe
    nghe_tu = max(0.0, bat_dau - 0.1)
    nghe_den = min(
        len(am_thanh) / tan_so_nghe,
        ket_thuc + 0.1,
    )

    doan_am_thanh = am_thanh[
        int(nghe_tu * tan_so_nghe):
        int(nghe_den * tan_so_nghe)
    ]

    print(
        f"\n{ten_doan}: "
        f"biên MFA {bat_dau:.3f}–{ket_thuc:.3f} giây"
    )

    display(Audio(
        doan_am_thanh,
        rate=tan_so_nghe,
        normalize=False,
    ))

# Nghe riêng hai từ đầu để kiểm tra ranh giới I / DONT
for vi_tri in [0, 1]:
    dong = bang_tu_mfa.iloc[vi_tri]

    doan = am_thanh[
        int(float(dong["start"]) * tan_so_nghe):
        int(float(dong["end"]) * tan_so_nghe)
    ]

    print(
        f"\nRiêng từ {dong['word']}: "
        f"{dong['start']:.3f}–{dong['end']:.3f} giây"
    )

    display(Audio(
        doan,
        rate=tan_so_nghe,
        normalize=False,
    ))

TOÀN BỘ CLIP
Transcript: I DONT KNOW CLASSIC SUPERHERO KIND OF GETTING ON KIND OF THING



I DONT KNOW: biên MFA 0.060–0.840 giây



CLASSIC SUPERHERO KIND OF: biên MFA 0.840–2.360 giây



ON KIND OF THING: biên MFA 3.230–4.120 giây



Riêng từ i: 0.060–0.090 giây



Riêng từ dont: 0.090–0.780 giây


In [ ]:
from IPython.display import Audio, display

doan_dau = am_thanh[:int(1.3 * tan_so_nghe)]

print("1,3 giây đầu — tốc độ gốc")
display(Audio(
    doan_dau,
    rate=tan_so_nghe,
    normalize=False,
))

print("1,3 giây đầu — phát chậm 0,75 lần")
display(Audio(
    doan_dau,
    rate=int(tan_so_nghe * 0.75),
    normalize=False,
))

1,3 giây đầu — tốc độ gốc


1,3 giây đầu — phát chậm 0,75 lần


In [ ]:
from pathlib import Path
import pandas as pd
import wave
import shutil
from datetime import datetime, timezone

# Đọc lại đúng danh sách pilot
pilot = pd.read_csv(
    thu_muc_du_lieu
    / "preprocessing"
    / "shared"
    / "pilot_manifest.csv",
    dtype={
        "id": "string",
        "video_id": "string",
        "clip_id": "string",
    },
    keep_default_na=False,
)

assert len(pilot) == 10, "Pilot chưa đủ 10 mẫu"
assert pilot["id"].is_unique, "Pilot có ID trùng"

ma_pilot_mfa = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

thu_muc_pilot_mfa = (
    thu_muc_du_lieu
    / "preprocessing"
    / "alignment"
    / "pilot"
    / ma_pilot_mfa
)

thu_muc_corpus_pilot = thu_muc_pilot_mfa / "corpus"
thu_muc_corpus_pilot.mkdir(parents=True, exist_ok=False)

nhat_ky_chuan_bi = []

for _, mau in pilot.iterrows():
    id_mau = mau["id"]
    wav_nguon = thu_muc_wav / f"{id_mau}.wav"

    try:
        assert wav_nguon.is_file(), "Không tìm thấy WAV"

        with wave.open(str(wav_nguon), "rb") as tep:
            assert tep.getnchannels() == 1, "WAV chưa phải mono"
            assert tep.getframerate() == 16000, "Sai sample rate"
            assert tep.getsampwidth() == 2, "WAV chưa đúng 16-bit"
            assert tep.getcomptype() == "NONE", "WAV không phải PCM"
            assert tep.getnframes() > 0, "WAV rỗng"

            thoi_luong_mau = (
                tep.getnframes() / tep.getframerate()
            )

        # Lấy transcript từ manifest chung theo ID
        dong_manifest = manifest.loc[manifest["id"] == id_mau]
        assert len(dong_manifest) == 1, "ID trong manifest không duy nhất"

        raw_text = dong_manifest.iloc[0]["raw_text"]
        text_lab = " ".join(raw_text.split())

        assert text_lab, "Transcript trống"
        assert raw_text == mau["raw_text"], (
            "Transcript pilot khác manifest"
        )
        assert (
            dong_manifest.iloc[0]["video_id"] == mau["video_id"]
        ), "Video ID pilot khác manifest"

        thu_muc_nguoi_noi = (
            thu_muc_corpus_pilot / str(mau["video_id"])
        )
        thu_muc_nguoi_noi.mkdir(parents=True, exist_ok=True)

        wav_dich = thu_muc_nguoi_noi / f"{id_mau}.wav"
        lab_dich = thu_muc_nguoi_noi / f"{id_mau}.lab"

        shutil.copyfile(wav_nguon, wav_dich)

        assert wav_dich.read_bytes() == wav_nguon.read_bytes(), (
            "WAV sao chép không khớp"
        )

        lab_dich.write_text(
            text_lab + "\n",
            encoding="utf-8",
        )

        nhat_ky_chuan_bi.append({
            "id": id_mau,
            "status": "success",
            "duration": thoi_luong_mau,
            "word_count": len(text_lab.split()),
            "error": "",
        })

    except Exception as loi:
        nhat_ky_chuan_bi.append({
            "id": id_mau,
            "status": "failed",
            "duration": None,
            "word_count": None,
            "error": f"{type(loi).__name__}: {loi}",
        })

bang_chuan_bi = pd.DataFrame(nhat_ky_chuan_bi)

bang_chuan_bi.to_csv(
    thu_muc_pilot_mfa / "preparation_log.csv",
    index=False,
    encoding="utf-8-sig",
)

display(bang_chuan_bi)

pilot_san_sang = bool(
    bang_chuan_bi["status"].eq("success").all()
)

print("Corpus:", thu_muc_corpus_pilot)
print("Đủ 10 mẫu sẵn sàng:", pilot_san_sang)

,id,status,duration,word_count,error
0,7JsX8y1ysxY$_$12,success,5.234687,12,
1,8OtFthrtaJM$_$23,success,8.671188,24,
2,8d-gEyoeBzc$_$5,success,5.606187,19,
3,9T9Hf74oK10$_$9,success,5.791938,20,
4,Bfr499ggo-0$_$12,success,2.587563,9,
5,I5y0__X72p0$_$35,success,2.192875,6,
6,I5y0__X72p0$_$6,success,3.957562,18,
7,Nzq88NnDkEk$_$32,success,3.353813,9,
8,VbQk4H8hgr0$_$24,success,2.494687,8,
9,Vj1wYRQjB-o$_$2,success,4.816687,21,


Corpus: /content/drive/MyDrive/CMU-MOSI/preprocessing/alignment/pilot/20261102T093614_520589Z/corpus
Đủ 10 mẫu sẵn sàng: True


In [ ]:
assert pilot_san_sang, "Còn mẫu chưa chuẩn bị thành công"
assert tu_dien_bo_sung.is_file(), "Không tìm thấy từ điển riêng"

ma_oov_pilot = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

thu_muc_oov_pilot = (
    thu_muc_pilot_mfa / f"oov_check_{ma_oov_pilot}"
)

thu_muc_tam_oov_pilot = Path(
    f"/content/mfa_oov_pilot_{ma_oov_pilot}"
)

print("Đang kiểm tra OOV trên 10 clip pilot...", flush=True)

ket_qua_oov_pilot = subprocess.run(
    lenh_mfa + [
        "find_oovs",
        str(thu_muc_corpus_pilot),
        str(tu_dien_bo_sung),
        str(thu_muc_oov_pilot),
        "--temporary_directory", str(thu_muc_tam_oov_pilot),
        "--num_jobs", "1",
    ],
    capture_output=True,
    text=True,
)

# Giữ log cả khi lệnh thất bại
thu_muc_oov_pilot.mkdir(parents=True, exist_ok=True)

(thu_muc_oov_pilot / "command_output.log").write_text(
    ket_qua_oov_pilot.stdout + "\n" + ket_qua_oov_pilot.stderr,
    encoding="utf-8",
)

print(ket_qua_oov_pilot.stdout)
print(ket_qua_oov_pilot.stderr)

ket_qua_oov_pilot.check_returncode()

print("\nCÁC BÁO CÁO OOV")

for tep in sorted(thu_muc_oov_pilot.rglob("*")):
    if tep.is_file() and tep.suffix in {".txt", ".csv"}:
        print("\nFile:", tep.name)

        noi_dung = tep.read_text(encoding="utf-8")
        print(noi_dung if noi_dung.strip() else "(file trống)")

Đang kiểm tra OOV trên 10 clip pilot...
  10% ━━━╸                                10/100  [ 0:00:01 < -:--:-- , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:01 < 0:00:00 , ? it/s ]

 INFO     Setting up corpus information...                                      
 INFO     Loading corpus from source files...                                   
 INFO     Found 9 speakers across 10 files, average number of utterances per    
          speaker: 1.1111111111111112                                           
 INFO     Initializing multiprocessing jobs...                                  
 INFO     Normalizing text...                                       

In [ ]:
cac_tep_oov = list(
    thu_muc_oov_pilot.glob("oovs_found_*.txt")
)

assert len(cac_tep_oov) == 1, (
    "Cần xác định duy nhất một file danh sách OOV"
)

tep_oov_pilot = cac_tep_oov[0]

tu_thieu_pilot = [
    dong.strip()
    for dong in tep_oov_pilot.read_text(encoding="utf-8").splitlines()
    if dong.strip()
]

print("Từ cần bổ sung:", tu_thieu_pilot)

assert set(tu_thieu_pilot) == {"creepiest", "shouldve"}, (
    "Danh sách OOV khác kết quả vừa kiểm tra"
)

ma_g2p_pilot = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

thu_muc_g2p_pilot = (
    thu_muc_pilot_mfa / f"g2p_{ma_g2p_pilot}"
)

thu_muc_g2p_pilot.mkdir(parents=True, exist_ok=False)

tep_phat_am_pilot = (
    thu_muc_g2p_pilot / "pilot_oov_pronunciations.dict"
)

g2p_pilot = subprocess.run(
    lenh_mfa + [
        "g2p",
        str(tep_oov_pilot),
        "english_us_arpa",
        str(tep_phat_am_pilot),
    ],
    capture_output=True,
    text=True,
)

(thu_muc_g2p_pilot / "g2p_output.log").write_text(
    g2p_pilot.stdout + "\n" + g2p_pilot.stderr,
    encoding="utf-8",
)

print(g2p_pilot.stdout)
print(g2p_pilot.stderr)

g2p_pilot.check_returncode()

assert tep_phat_am_pilot.is_file(), "Chưa tạo được file phát âm"

print("\nCÁCH PHÁT ÂM ĐỀ XUẤT")
print(tep_phat_am_pilot.read_text(encoding="utf-8"))

Từ cần bổ sung: ['creepiest', 'shouldve']
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2  [ 0:00:01 < 0:00:00 , ? it/s ]

 INFO     Generating pronunciations...                                          
 INFO     Done! Everything took 3.933 seconds                                   


CÁCH PHÁT ÂM ĐỀ XUẤT
creepiest	K R IY1 P IY0 AH0 S T
shouldve	SH UH1 D V



In [ ]:
tu_dien_pilot = thu_muc_g2p_pilot / "english_us_arpa_pilot.dict"

# Sao chép từ điển đã có superhero, không ghi đè file cũ
with tu_dien_bo_sung.open("rb") as nguon:
    with tu_dien_pilot.open("xb") as dich:
        shutil.copyfileobj(nguon, dich)

them_tu_pilot = subprocess.run(
    lenh_mfa + [
        "model", "add_words",
        str(tu_dien_pilot),
        str(tep_phat_am_pilot),
    ],
    capture_output=True,
    text=True,
)

print(them_tu_pilot.stdout)
print(them_tu_pilot.stderr)

them_tu_pilot.check_returncode()

# Kiểm tra cả ba mục đã bổ sung
cac_tu_can_co = {"superhero", "creepiest", "shouldve"}
cac_tu_da_co = set()

with tu_dien_pilot.open("r", encoding="utf-8") as tep:
    for dong in tep:
        thanh_phan = dong.split()

        if thanh_phan and thanh_phan[0].lower() in cac_tu_can_co:
            cac_tu_da_co.add(thanh_phan[0].lower())
            print(dong.strip())

assert cac_tu_da_co == cac_tu_can_co, "Còn thiếu mục phát âm"

print("\nTừ điển pilot:", tu_dien_pilot)



creepiest	K R IY1 P IY0 AH0 S T
shouldve	SH UH1 D V
superhero	S UW2 P ER0 HH IH1 R OW0
superhero	S UW2 P ER0 HH IY1 R OW0

Từ điển pilot: /content/drive/MyDrive/CMU-MOSI/preprocessing/alignment/pilot/20261102T093614_520589Z/g2p_20261102T093720_323868Z/english_us_arpa_pilot.dict


In [ ]:
ma_oov_lai = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

thu_muc_oov_pilot_lai = (
    thu_muc_pilot_mfa / f"oov_recheck_{ma_oov_lai}"
)

kiem_tra_lai = subprocess.run(
    lenh_mfa + [
        "find_oovs",
        str(thu_muc_corpus_pilot),
        str(tu_dien_pilot),
        str(thu_muc_oov_pilot_lai),
        "--temporary_directory",
        f"/content/mfa_pilot_recheck_{ma_oov_lai}",
        "--num_jobs", "1",
    ],
    capture_output=True,
    text=True,
)

thu_muc_oov_pilot_lai.mkdir(parents=True, exist_ok=True)

(thu_muc_oov_pilot_lai / "command_output.log").write_text(
    kiem_tra_lai.stdout + "\n" + kiem_tra_lai.stderr,
    encoding="utf-8",
)

print(kiem_tra_lai.stdout)
print(kiem_tra_lai.stderr)

kiem_tra_lai.check_returncode()

for tep in sorted(thu_muc_oov_pilot_lai.rglob("*.txt")):
    print("\nFile:", tep.name)
    noi_dung = tep.read_text(encoding="utf-8")
    print(noi_dung if noi_dung.strip() else "(file trống)")

  10% ━━━╸                                10/100  [ 0:00:01 < -:--:-- , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:01 < 0:00:00 , ? it/s ]

 INFO     Setting up corpus information...                                      
 INFO     Loading corpus from source files...                                   
 INFO     Found 9 speakers across 10 files, average number of utterances per    
          speaker: 1.1111111111111112                                           
 INFO     Initializing multiprocessing jobs...                                  
 INFO     Normalizing text...                                                   
 INFO     Generating MFCCs.

In [ ]:
import json
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path

assert pilot_san_sang, "Pilot chưa chuẩn bị đầy đủ"
assert tu_dien_pilot.is_file(), "Không tìm thấy từ điển pilot"
assert kiem_tra_lai.returncode == 0, "Bước kiểm tra OOV chưa thành công"

ma_align_pilot = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

thu_muc_align_pilot = (
    thu_muc_pilot_mfa / f"alignment_{ma_align_pilot}"
)

thu_muc_align_pilot.mkdir(parents=True, exist_ok=False)

thu_muc_textgrid_pilot = thu_muc_align_pilot / "textgrids"

lenh_align_pilot = lenh_mfa + [
    "align",
    str(thu_muc_corpus_pilot),
    str(tu_dien_pilot),
    "english_us_arpa",
    str(thu_muc_textgrid_pilot),
    "--temporary_directory",
    f"/content/mfa_align_pilot_{ma_align_pilot}",
    "--num_jobs", "1",
]

# Lưu cấu hình để truy lại đúng lần chạy
cau_hinh_pilot = {
    "mfa_version": "3.4.3",
    "acoustic_model": "english_us_arpa",
    "dictionary_path": str(tu_dien_pilot),
    "dictionary_sha256": hashlib.sha256(
        tu_dien_pilot.read_bytes()
    ).hexdigest(),
    "pilot_ids": pilot["id"].tolist(),
    "command": lenh_align_pilot,
}

(thu_muc_align_pilot / "config.json").write_text(
    json.dumps(cau_hinh_pilot, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Đang chạy alignment cho 10 clip pilot...", flush=True)

align_pilot = subprocess.run(
    lenh_align_pilot,
    capture_output=True,
    text=True,
)

(thu_muc_align_pilot / "command_output.log").write_text(
    align_pilot.stdout + "\n" + align_pilot.stderr,
    encoding="utf-8",
)

print(align_pilot.stdout)
print(align_pilot.stderr)

# Ghi nhận đầu ra cho từng ID, kể cả khi lệnh lỗi
cac_textgrid_pilot = list(
    thu_muc_textgrid_pilot.rglob("*.TextGrid")
)

nhat_ky_align = []

for id_mau in pilot["id"]:
    cac_tep = [
        tep for tep in cac_textgrid_pilot
        if tep.stem == id_mau
    ]

    if len(cac_tep) == 1 and cac_tep[0].stat().st_size > 0:
        trang_thai = "textgrid_created"
        duong_dan = str(cac_tep[0])
        loi = ""
    else:
        trang_thai = "needs_check"
        duong_dan = ""
        loi = (
            f"Tìm thấy {len(cac_tep)} TextGrid; "
            "cần kiểm tra file thiếu, trùng hoặc rỗng"
        )

    nhat_ky_align.append({
        "id": id_mau,
        "status": trang_thai,
        "textgrid_path": duong_dan,
        "error": loi,
    })

bang_align_pilot = pd.DataFrame(nhat_ky_align)

bang_align_pilot.to_csv(
    thu_muc_align_pilot / "alignment_log.csv",
    index=False,
    encoding="utf-8-sig",
)

display(bang_align_pilot)

print("Mã kết thúc lệnh:", align_pilot.returncode)
print("Thư mục kết quả:", thu_muc_align_pilot)

align_pilot.check_returncode()

Đang chạy alignment cho 10 clip pilot...
  10% ━━━╸                                10/100  [ 0:00:01 < -:--:-- , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:02 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:04 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:00 < 0:00:00 , 57 it/s ]

 INFO     Setting up corpus information...                         

,id,status,textgrid_path,error
0,7JsX8y1ysxY$_$12,textgrid_created,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,
1,8OtFthrtaJM$_$23,textgrid_created,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,
2,8d-gEyoeBzc$_$5,textgrid_created,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,
3,9T9Hf74oK10$_$9,textgrid_created,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,
4,Bfr499ggo-0$_$12,textgrid_created,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,
5,I5y0__X72p0$_$35,textgrid_created,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,
6,I5y0__X72p0$_$6,textgrid_created,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,
7,Nzq88NnDkEk$_$32,textgrid_created,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,
8,VbQk4H8hgr0$_$24,textgrid_created,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,
9,Vj1wYRQjB-o$_$2,textgrid_created,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,


Mã kết thúc lệnh: 0
Thư mục kết quả: /content/drive/MyDrive/CMU-MOSI/preprocessing/alignment/pilot/20261102T093614_520589Z/alignment_20261102T093905_856138Z


In [ ]:
import json
import math
import subprocess
import wave
import pandas as pd

# Đọc tất cả TextGrid bằng praatio trong môi trường MFA
code_doc_pilot = r"""
import json
import sys
from pathlib import Path
from praatio import textgrid

ket_qua = {}

for tep in Path(sys.argv[1]).rglob("*.TextGrid"):
    tg = textgrid.openTextgrid(
        str(tep),
        includeEmptyIntervals=True,
    )

    if "words" not in tg.tierNames:
        raise ValueError(f"Không thấy tier words: {tep}")

    if tep.stem in ket_qua:
        raise ValueError(f"Trùng ID TextGrid: {tep.stem}")

    ket_qua[tep.stem] = [
        {"start": float(s), "end": float(e), "word": w}
        for s, e, w in tg.getTier("words").entries
        if w.strip()
    ]

print(json.dumps(ket_qua, ensure_ascii=False))
"""

doc_pilot = subprocess.run(
    [
        str(micromamba), "run", "-p", str(mfa_env),
        "python", "-c", code_doc_pilot,
        str(thu_muc_textgrid_pilot),
    ],
    capture_output=True,
    text=True,
    check=True,
)

tu_mfa_pilot = json.loads(doc_pilot.stdout)
nhat_ky_kiem_tra = []

for id_mau in pilot["id"].astype(str):
    # Tìm transcript và WAV đã dùng để chạy MFA
    cac_lab = list(thu_muc_corpus_pilot.rglob(f"{id_mau}.lab"))
    assert len(cac_lab) == 1, f"Transcript thiếu hoặc trùng: {id_mau}"

    tep_lab = cac_lab[0]
    tep_wav = tep_lab.with_suffix(".wav")
    words_goc = tep_lab.read_text(encoding="utf-8").split()

    with wave.open(str(tep_wav), "rb") as f:
        duration = f.getnframes() / f.getframerate()

    cac_tu = tu_mfa_pilot.get(id_mau, [])
    words_mfa = [x["word"] for x in cac_tu]

    khop_tu = (
        [w.casefold() for w in words_goc]
        == [w.casefold() for w in words_mfa]
    )

    loi = []
    if not cac_tu:
        loi.append("Thiếu TextGrid hoặc không có từ")

    end_truoc = 0.0
    for vi_tri, x in enumerate(cac_tu):
        s, e = x["start"], x["end"]

        if not (math.isfinite(s) and math.isfinite(e)):
            loi.append(f"Từ {vi_tri}: thời gian không hữu hạn")
            continue

        if not (0 <= s < e <= duration + 0.001):
            loi.append(f"Từ {vi_tri}: khoảng thời gian không hợp lệ")

        if s < end_truoc - 0.000001:
            loi.append(f"Từ {vi_tri}: chồng lấn hoặc sai thứ tự")

        end_truoc = e

    nhat_ky_kiem_tra.append({
        "id": id_mau,
        "word_count_original": len(words_goc),
        "word_count_mfa": len(words_mfa),
        "word_sequence_match": khop_tu,
        "timestamp_structure_pass": not loi,
        "listening_review": (
            "needs_review"
            if id_mau == "7JsX8y1ysxY$_$12"
            else "not_reviewed"
        ),
        "error": "; ".join(loi),
    })

bang_kiem_tra_pilot = pd.DataFrame(nhat_ky_kiem_tra)

bang_kiem_tra_pilot.to_csv(
    thu_muc_align_pilot / "word_timestamp_validation.csv",
    index=False,
    encoding="utf-8-sig",
)

(thu_muc_align_pilot / "word_intervals.json").write_text(
    json.dumps(tu_mfa_pilot, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

display(bang_kiem_tra_pilot)

print(
    "Số clip vượt kiểm tra tự động:",
    (
        bang_kiem_tra_pilot["word_sequence_match"]
        & bang_kiem_tra_pilot["timestamp_structure_pass"]
    ).sum(),
    "/", len(pilot),
)

,id,word_count_original,word_count_mfa,word_sequence_match,timestamp_structure_pass,listening_review,error
0,7JsX8y1ysxY$_$12,12,12,True,True,needs_review,
1,8OtFthrtaJM$_$23,24,24,True,True,not_reviewed,
2,8d-gEyoeBzc$_$5,19,19,True,True,not_reviewed,
3,9T9Hf74oK10$_$9,20,20,True,True,not_reviewed,
4,Bfr499ggo-0$_$12,9,9,True,True,not_reviewed,
5,I5y0__X72p0$_$35,6,6,True,True,not_reviewed,
6,I5y0__X72p0$_$6,18,18,True,True,not_reviewed,
7,Nzq88NnDkEk$_$32,9,9,True,True,not_reviewed,
8,VbQk4H8hgr0$_$24,8,8,True,True,not_reviewed,
9,Vj1wYRQjB-o$_$2,21,21,True,True,not_reviewed,


Số clip vượt kiểm tra tự động: 10 / 10


In [6]:
import json
import pandas as pd
from pathlib import Path

goc = Path("/content/drive/MyDrive/CMU-MOSI")

thu_muc_align_pilot = (
    goc
    / "preprocessing/alignment/pilot/20261102T093614_520589Z"
    / "alignment_20261102T093905_856138Z"
)

tep_text = (
    goc
    / "preprocessing/text/pilot/20261102T092142_188135Z"
    / "text_pilot_results.json"
)

tep_khoang_tu = thu_muc_align_pilot / "word_intervals.json"
tep_kiem_tra = thu_muc_align_pilot / "word_timestamp_validation.csv"

for tep in [tep_text, tep_khoang_tu, tep_kiem_tra]:
    assert tep.is_file(), f"Không tìm thấy: {tep}"

ket_qua_pilot = json.loads(
    tep_text.read_text(encoding="utf-8")
)

tu_mfa_pilot = json.loads(
    tep_khoang_tu.read_text(encoding="utf-8")
)

bang_kiem_tra_pilot = pd.read_csv(
    tep_kiem_tra,
    dtype={"id": "string"},
    keep_default_na=False,
)

# Đọc rõ các giá trị True/False từ CSV
for cot in ["word_sequence_match", "timestamp_structure_pass"]:
    gia_tri = bang_kiem_tra_pilot[cot].astype(str).str.lower()
    assert gia_tri.isin(["true", "false"]).all(), cot
    bang_kiem_tra_pilot[cot] = gia_tri.eq("true")

pilot = pd.read_csv(
    goc / "preprocessing/shared/pilot_manifest.csv",
    dtype={"id": "string", "video_id": "string", "clip_id": "string"},
    keep_default_na=False,
)

assert len(pilot) == 10
assert pilot["id"].is_unique
assert bang_kiem_tra_pilot["id"].is_unique

assert (
    set(pilot["id"])
    == set(ket_qua_pilot)
    == set(tu_mfa_pilot)
    == set(bang_kiem_tra_pilot["id"])
)

print("Đã nạp đủ dữ liệu của 10 mẫu pilot.")

Đã nạp đủ dữ liệu của 10 mẫu pilot.


In [7]:
import json
import pandas as pd

import importlib.util
from pathlib import Path

tep_alignment = Path(
    "/content/drive/MyDrive/CMU-MOSI"
    "/src/preprocessing/alignment.py"
)

dac_ta = importlib.util.spec_from_file_location(
    "mosi_alignment",
    tep_alignment,
)

alignment_module = importlib.util.module_from_spec(dac_ta)
dac_ta.loader.exec_module(alignment_module)

ket_qua_ghep_pilot = {}
nhat_ky_ghep = []

kiem_tra_theo_id = bang_kiem_tra_pilot.set_index("id")

for id_mau in pilot["id"].astype(str):
    text = ket_qua_pilot[id_mau]
    cac_tu = tu_mfa_pilot[id_mau]
    kiem_tra = kiem_tra_theo_id.loc[id_mau]

    assert bool(kiem_tra["word_sequence_match"]), id_mau
    assert bool(kiem_tra["timestamp_structure_pass"]), id_mau

    words = text["words"]
    word_ids = text["word_ids"]

    assert [w.casefold() for w in words] == [
        x["word"].casefold() for x in cac_tu
    ], f"Không khớp từ: {id_mau}"

    so_token = len(text["input_ids"])

    for truong in ["attention_mask", "token_type_ids", "word_ids"]:
        assert len(text[truong]) == so_token, (
            f"Sai độ dài {truong}: {id_mau}"
        )

    assert set(word_ids) - {-1} == set(range(len(words))), (
        f"word_ids thiếu từ hoặc ngoài phạm vi: {id_mau}"
    )

    ket_qua_alignment = alignment_module.process_alignment(
        word_intervals=cac_tu,
        words=words,
        word_ids=word_ids,
    )

    word_start = ket_qua_alignment["word_start"]
    word_end = ket_qua_alignment["word_end"]
    token_start = ket_qua_alignment["token_start"]
    token_end = ket_qua_alignment["token_end"]
    token_time_mask = ket_qua_alignment["token_time_mask"]

    ket_qua_ghep_pilot[id_mau] = {
        **text,
        "word_start": word_start,
        "word_end": word_end,
        "token_start": token_start,
        "token_end": token_end,
        "token_time_mask": token_time_mask,
        "listening_review": kiem_tra["listening_review"],
    }

    nhat_ky_ghep.append({
        "id": id_mau,
        "word_count": len(words),
        "token_count": so_token,
        "tokens_with_time": sum(token_time_mask),
        "status": "mapped",
        "listening_review": kiem_tra["listening_review"],
    })

# Lưu thành file riêng, giữ nguyên kết quả Text ban đầu
tep_ghep = thu_muc_align_pilot / "text_token_timestamps.json"

tep_ghep.write_text(
    json.dumps(
        ket_qua_ghep_pilot,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    ),
    encoding="utf-8",
)

bang_ghep = pd.DataFrame(nhat_ky_ghep)
bang_ghep.to_csv(
    thu_muc_align_pilot / "token_mapping_log.csv",
    index=False,
    encoding="utf-8-sig",
)

display(bang_ghep)
print("Đã lưu:", tep_ghep)

# Xem cách ghép của clip đầu
id_mau = str(pilot.iloc[0]["id"])
mau = ket_qua_ghep_pilot[id_mau]

display(pd.DataFrame({
    "input_id": mau["input_ids"],
    "word_id": mau["word_ids"],
    "word": [
        mau["words"][i] if i != -1 else "(token đặc biệt)"
        for i in mau["word_ids"]
    ],
    "start": mau["token_start"],
    "end": mau["token_end"],
    "time_mask": mau["token_time_mask"],
}))

,id,word_count,token_count,tokens_with_time,status,listening_review
0,7JsX8y1ysxY$_$12,12,15,13,mapped,needs_review
1,8OtFthrtaJM$_$23,24,30,28,mapped,not_reviewed
2,8d-gEyoeBzc$_$5,19,22,20,mapped,not_reviewed
3,9T9Hf74oK10$_$9,20,22,20,mapped,not_reviewed
4,Bfr499ggo-0$_$12,9,12,10,mapped,not_reviewed
5,I5y0__X72p0$_$35,6,8,6,mapped,not_reviewed
6,I5y0__X72p0$_$6,18,20,18,mapped,not_reviewed
7,Nzq88NnDkEk$_$32,9,11,9,mapped,not_reviewed
8,VbQk4H8hgr0$_$24,8,10,8,mapped,not_reviewed
9,Vj1wYRQjB-o$_$2,21,23,21,mapped,not_reviewed


Đã lưu: /content/drive/MyDrive/CMU-MOSI/preprocessing/alignment/pilot/20261102T093614_520589Z/alignment_20261102T093905_856138Z/text_token_timestamps.json


,input_id,word_id,word,start,end,time_mask
0,101,-1,(token đặc biệt),NaN,NaN,False
1,1045,0,I,0.06,0.09,True
2,2123,1,DONT,0.09,0.78,True
3,2102,1,DONT,0.09,0.78,True
4,2113,2,KNOW,0.78,0.84,True
5,4438,3,CLASSIC,0.84,1.19,True
6,16251,4,SUPERHERO,1.33,1.99,True
7,2785,5,KIND,1.99,2.26,True
8,1997,6,OF,2.26,2.36,True
9,2893,7,GETTING,2.91,3.23,True


In [5]:
cac_bien_can = [
    "pilot",
    "ket_qua_pilot",
    "tu_mfa_pilot",
    "bang_kiem_tra_pilot",
    "thu_muc_align_pilot",
    "alignment_module",
]

for ten in cac_bien_can:
    print(ten, ":", ten in globals())

pilot : False
ket_qua_pilot : False
tu_mfa_pilot : False
bang_kiem_tra_pilot : False
thu_muc_align_pilot : False
alignment_module : True


In [ ]:
from IPython.display import Audio, display

for id_mau, tu_can_nghe in [
    ("8OtFthrtaJM$_$23", "creepiest"),
    ("Bfr499ggo-0$_$12", "shouldve"),
]:
    cac_wav = list(
        thu_muc_corpus_pilot.rglob(f"{id_mau}.wav")
    )
    assert len(cac_wav) == 1, f"WAV thiếu hoặc trùng: {id_mau}"

    mau = ket_qua_ghep_pilot[id_mau]

    print("\nID:", id_mau)
    print("Transcript:", mau["clean_text"])

    display(Audio(filename=str(cac_wav[0])))

    display(pd.DataFrame([
        {
            "word": word,
            "start": mau["word_start"][i],
            "end": mau["word_end"][i],
        }
        for i, word in enumerate(mau["words"])
        if word.casefold() == tu_can_nghe
    ]))


ID: 8OtFthrtaJM$_$23
Transcript: BUT HES JUST THE CREEPIEST OLD GUY YOULL EVER SEE IN YOUR LIFE HE STANDS OVER PEOPLE WHEN THEYRE ASLEEP JUST STARING AT THEM


,word,start,end
0,CREEPIEST,1.03,1.93



ID: Bfr499ggo-0$_$12
Transcript: THEY SHOULDVE JUST CUT THE WEDDING IN THE BEGINNING


,word,start,end
0,SHOULDVE,0.15,0.42


In [ ]:

# Giữ các ghi nhận đã có nếu chạy lại cell

da_nghe = {
    "7JsX8y1ysxY$_$12",
    "8OtFthrtaJM$_$23",
    "Bfr499ggo-0$_$12",
}

for id_mau in pilot["id"].astype(str):
    if id_mau in da_nghe:
        continue

    cac_wav = list(thu_muc_corpus_pilot.rglob(f"{id_mau}.wav"))
    assert len(cac_wav) == 1, f"WAV thiếu hoặc trùng: {id_mau}"

    print("\nID:", id_mau)
    print("Transcript:", ket_qua_ghep_pilot[id_mau]["clean_text"])
    display(Audio(filename=str(cac_wav[0])))


ID: 8d-gEyoeBzc$_$5
Transcript: UM AND IM GONNA REALLY POINT OUT WHY THAT I DONT THINK HE REALLY PULLED HE PULLED THAT OFF



ID: 9T9Hf74oK10$_$9
Transcript: UM IN A WAY A LOT OF THE THEMES IN NEVER LET ME GO WHICH WERE VERY PROFOUND AND DEEP



ID: I5y0__X72p0$_$35
Transcript: BUT IT WAS IT WAS INTERESTING



ID: I5y0__X72p0$_$6
Transcript: THEY S AND ALL THE PEOPLE WALKING OUT OF THE THEATRE AND BEYOND THE TRAILER REALLY LIKED IT



ID: Nzq88NnDkEk$_$32
Transcript: AND UM LOOKING REALLY FORWARD TO SEEING THAT MOVIE



ID: VbQk4H8hgr0$_$24
Transcript: UM IT WAS A WASTE OF MY TIME



ID: Vj1wYRQjB-o$_$2
Transcript: HE ALSO HAS SOME REALLY COOL GUNS SO THAT ITS LIKE A DESERT EAGLE BUT IT HAS TO BARRELS TO IT


In [ ]:
import json
import pandas as pd


# Hai clip đã nghe ở bước trước

import wave
import numpy as np
from IPython.display import Audio, display

for id_mau, tu_can_nghe in [
    ("8OtFthrtaJM$_$23", "creepiest"),
    ("Bfr499ggo-0$_$12", "shouldve"),
]:
    tep_wav = next(thu_muc_corpus_pilot.rglob(f"{id_mau}.wav"))

    with wave.open(str(tep_wav), "rb") as f:
        assert f.getnchannels() == 1
        assert f.getsampwidth() == 2
        sr = f.getframerate()
        am_thanh = np.frombuffer(
            f.readframes(f.getnframes()), dtype="<i2"
        ).astype(np.float32) / 32768.0

    mau = ket_qua_ghep_pilot[id_mau]

    for i, word in enumerate(mau["words"]):
        if word.casefold() != tu_can_nghe:
            continue

        start = mau["word_start"][i]
        end = mau["word_end"][i]

        print(f"\nID: {id_mau}")
        print(f"Từ: {word} | MFA: {start:.3f}–{end:.3f} giây")

        for ten, dem in [
            ("Đúng khoảng MFA", 0),
            ("Kèm ngữ cảnh hai bên", 0.2),
        ]:
            dau = max(0, int((start - dem) * sr))
            cuoi = min(len(am_thanh), int((end + dem) * sr))

            print(ten)
            display(Audio(
                am_thanh[dau:cuoi],
                rate=sr,
                normalize=False,
            ))


ID: 8OtFthrtaJM$_$23
Từ: CREEPIEST | MFA: 1.030–1.930 giây
Đúng khoảng MFA


Kèm ngữ cảnh hai bên



ID: Bfr499ggo-0$_$12
Từ: SHOULDVE | MFA: 0.150–0.420 giây
Đúng khoảng MFA


Kèm ngữ cảnh hai bên


9 clip khớp transcript; clip 7JsX8y1ysxY$_$12 cần xem lại cụm đầu. Hai từ creepiest và shouldve được nghe đủ trong khoảng MFA. Không tự áp dụng đánh giá nghe cũ cho lần chạy mới.

---



In [2]:
from pathlib import Path

tep_alignment = Path(
    "/content/drive/MyDrive/CMU-MOSI"
    "/src/preprocessing/alignment.py"
)

noi_dung_module = '''
"""Kiểm tra khoảng từ MFA và ánh xạ timestamp sang token."""

import math
from numbers import Integral


def process_alignment(word_intervals, words, word_ids, duration=None):
    """Trả timestamp khi từ MFA khớp words; báo lỗi nếu không khớp."""

    words = list(words)
    word_ids = list(word_ids)

    if not words:
        raise ValueError("Danh sách words trống")

    if any(not isinstance(w, str) or not w.strip() for w in words):
        raise ValueError("words phải chứa các từ không trống")

    if duration is not None:
        if not math.isfinite(duration) or duration <= 0:
            raise ValueError("Thời lượng clip không hợp lệ")

    # Bỏ khoảng im lặng, giữ nguyên các nhãn từ
    cac_khoang = []

    for khoang in word_intervals:
        tu = khoang["word"]

        if not isinstance(tu, str):
            raise ValueError("Nhãn từ MFA phải là chuỗi")

        if not tu.strip():
            continue

        cac_khoang.append({
            "word": tu,
            "start": float(khoang["start"]),
            "end": float(khoang["end"]),
        })

    words_mfa = [x["word"] for x in cac_khoang]

    if [w.casefold() for w in words] != [
        w.casefold() for w in words_mfa
    ]:
        raise ValueError(
            "Từ Text và MFA không khớp; không ghép timestamp. "
            f"Text: {words!r}; MFA: {words_mfa!r}"
        )

    end_truoc = 0.0

    for vi_tri, khoang in enumerate(cac_khoang):
        start = khoang["start"]
        end = khoang["end"]

        if not (math.isfinite(start) and math.isfinite(end)):
            raise ValueError(f"Từ {vi_tri}: thời gian không hữu hạn")

        if not 0 <= start < end:
            raise ValueError(f"Từ {vi_tri}: khoảng thời gian không hợp lệ")

        if duration is not None and end > duration + 0.001:
            raise ValueError(f"Từ {vi_tri}: vượt thời lượng clip")

        if start < end_truoc - 0.000001:
            raise ValueError(f"Từ {vi_tri}: chồng lấn hoặc sai thứ tự")

        end_truoc = end

    for chi_so in word_ids:
        if isinstance(chi_so, bool) or not isinstance(chi_so, Integral):
            raise ValueError("word_ids phải chứa chỉ số nguyên")

        if chi_so != -1 and not 0 <= chi_so < len(words):
            raise ValueError(f"word_id ngoài phạm vi: {chi_so}")

    chi_so_tu = [int(i) for i in word_ids if i != -1]

    if set(chi_so_tu) != set(range(len(words))):
        raise ValueError("word_ids chưa ánh xạ đủ các từ")

    if chi_so_tu != sorted(chi_so_tu):
        raise ValueError("word_ids không đúng thứ tự từ")

    word_start = [x["start"] for x in cac_khoang]
    word_end = [x["end"] for x in cac_khoang]

    token_start = []
    token_end = []
    token_time_mask = []

    for chi_so in word_ids:
        if chi_so == -1:
            token_start.append(None)
            token_end.append(None)
            token_time_mask.append(False)
        else:
            token_start.append(word_start[chi_so])
            token_end.append(word_end[chi_so])
            token_time_mask.append(True)

    return {
        "words": words,
        "word_ids": [int(i) for i in word_ids],
        "word_intervals": cac_khoang,
        "word_sequence_match": True,
        "timestamp_structure_pass": True,
        "word_start": word_start,
        "word_end": word_end,
        "token_start": token_start,
        "token_end": token_end,
        "token_time_mask": token_time_mask,
    }
'''

tep_alignment.parent.mkdir(parents=True, exist_ok=True)

# Sao lưu nếu file đã tồn tại và có nội dung khác
if tep_alignment.exists():
    noi_dung_cu = tep_alignment.read_text(encoding="utf-8")

    if noi_dung_cu != noi_dung_module:
        from datetime import datetime, timezone

        ma_sao_luu = datetime.now(timezone.utc).strftime(
            "%Y%m%dT%H%M%S_%fZ"
        )

        tep_sao_luu = tep_alignment.with_name(
            f"alignment_backup_{ma_sao_luu}.py"
        )
        tep_sao_luu.write_text(noi_dung_cu, encoding="utf-8")
        print("Đã sao lưu:", tep_sao_luu)

tep_alignment.write_text(noi_dung_module, encoding="utf-8")

print("Đã lưu module:", tep_alignment)

Đã lưu module: /content/drive/MyDrive/CMU-MOSI/src/preprocessing/alignment.py


In [3]:
import importlib.util

dac_ta = importlib.util.spec_from_file_location(
    "mosi_alignment",
    tep_alignment,
)

alignment_module = importlib.util.module_from_spec(dac_ta)
dac_ta.loader.exec_module(alignment_module)

# Một từ có hai subword; token đặc biệt không có timestamp
ket_qua_thu = alignment_module.process_alignment(
    word_intervals=[
        {"word": "", "start": 0.0, "end": 0.1},
        {"word": "dont", "start": 0.1, "end": 0.5},
    ],
    words=["DONT"],
    word_ids=[-1, 0, 0, -1],
    duration=0.6,
)

assert ket_qua_thu["token_start"] == [None, 0.1, 0.1, None]
assert ket_qua_thu["token_end"] == [None, 0.5, 0.5, None]
assert ket_qua_thu["token_time_mask"] == [False, True, True, False]

# Từ không khớp phải bị từ chối ghép
try:
    alignment_module.process_alignment(
        word_intervals=[
            {"word": "happy", "start": 0.1, "end": 0.5},
        ],
        words=["sad"],
        word_ids=[-1, 0, -1],
        duration=0.6,
    )
except ValueError as loi:
    print("Đã chặn trường hợp không khớp:", loi)
else:
    raise AssertionError("Module đã ghép sai khi từ không khớp")

print("Kiểm tra ánh xạ subword và token đặc biệt: PASS")
print("Kiểm tra từ không khớp: PASS")

Đã chặn trường hợp không khớp: Từ Text và MFA không khớp; không ghép timestamp. Text: ['sad']; MFA: ['happy']
Kiểm tra ánh xạ subword và token đặc biệt: PASS
Kiểm tra từ không khớp: PASS


In [4]:
import json
import pandas as pd
from pathlib import Path

# Thư mục alignment của lần chạy đã kiểm tra
thu_muc_ket_qua_cu = Path(
    "/content/drive/MyDrive/CMU-MOSI/preprocessing/alignment/pilot/"
    "20261102T093614_520589Z/alignment_20261102T093905_856138Z"
)

tep_khoang_tu = thu_muc_ket_qua_cu / "word_intervals.json"
tep_ghep_cu = thu_muc_ket_qua_cu / "text_token_timestamps.json"

assert tep_khoang_tu.is_file(), f"Không tìm thấy: {tep_khoang_tu}"
assert tep_ghep_cu.is_file(), f"Không tìm thấy: {tep_ghep_cu}"

khoang_tu_theo_id = json.loads(
    tep_khoang_tu.read_text(encoding="utf-8")
)

ket_qua_cu = json.loads(
    tep_ghep_cu.read_text(encoding="utf-8")
)

assert len(ket_qua_cu) == 10
assert set(ket_qua_cu) == set(khoang_tu_theo_id)

cac_truong_so_sanh = [
    "word_start",
    "word_end",
    "token_start",
    "token_end",
    "token_time_mask",
]

nhat_ky = []

for id_mau, ban_cu in ket_qua_cu.items():
    ban_moi = alignment_module.process_alignment(
        word_intervals=khoang_tu_theo_id[id_mau],
        words=ban_cu["words"],
        word_ids=ban_cu["word_ids"],
    )

    for truong in cac_truong_so_sanh:
        assert ban_moi[truong] == ban_cu[truong], (
            f"Khác kết quả tại {id_mau}, trường {truong}"
        )

    assert len(ban_moi["token_start"]) == len(ban_cu["input_ids"])

    nhat_ky.append({
        "id": id_mau,
        "word_count": len(ban_moi["words"]),
        "token_count": len(ban_moi["token_start"]),
        "match_old_result": True,
    })

bang_doi_chieu = pd.DataFrame(nhat_ky)

display(bang_doi_chieu)
print("Số mẫu khớp kết quả cũ:", len(bang_doi_chieu), "/ 10")

,id,word_count,token_count,match_old_result
0,7JsX8y1ysxY$_$12,12,15,True
1,8OtFthrtaJM$_$23,24,30,True
2,8d-gEyoeBzc$_$5,19,22,True
3,9T9Hf74oK10$_$9,20,22,True
4,Bfr499ggo-0$_$12,9,12,True
5,I5y0__X72p0$_$35,6,8,True
6,I5y0__X72p0$_$6,18,20,True
7,Nzq88NnDkEk$_$32,9,11,True
8,VbQk4H8hgr0$_$24,8,10,True
9,Vj1wYRQjB-o$_$2,21,23,True


Số mẫu khớp kết quả cũ: 10 / 10


In [8]:
from pathlib import Path
import subprocess

repo = Path("/content/DataMining-MSA")

if (repo / ".git").is_dir():
    subprocess.run(
        ["git", "status"],
        cwd=repo,
        check=True,
    )
    subprocess.run(
        ["git", "branch", "--show-current"],
        cwd=repo,
        check=True,
    )
else:
    print("Phiên này chưa có repository, cần clone lại.")

Phiên này chưa có repository, cần clone lại.


In [9]:
from pathlib import Path

tep_alignment = Path(
    "/content/drive/MyDrive/CMU-MOSI"
    "/src/preprocessing/alignment.py"
)

assert tep_alignment.is_file(), "Chưa có alignment.py"

phan_bo_sung = '''

def read_textgrid(textgrid_path):
    """Đọc các khoảng có từ trong tier words của MFA."""
    from pathlib import Path
    from praatio import textgrid

    tep = Path(textgrid_path)

    if not tep.is_file():
        raise FileNotFoundError(f"Không tìm thấy TextGrid: {tep}")

    tg = textgrid.openTextgrid(
        str(tep),
        includeEmptyIntervals=True,
    )

    if "words" not in tg.tierNames:
        raise ValueError(f"Không thấy tier words: {tep}")

    return [
        {"word": w, "start": float(s), "end": float(e)}
        for s, e, w in tg.getTier("words").entries
        if w.strip()
    ]


def align_textgrid(textgrid_path, words, word_ids, duration=None):
    """Đọc TextGrid, kiểm tra và ánh xạ thời gian sang token."""
    cac_khoang = read_textgrid(textgrid_path)

    return process_alignment(
        word_intervals=cac_khoang,
        words=words,
        word_ids=word_ids,
        duration=duration,
    )
'''

noi_dung = tep_alignment.read_text(encoding="utf-8")

if "def read_textgrid(" not in noi_dung:
    tep_alignment.write_text(
        noi_dung + phan_bo_sung,
        encoding="utf-8",
    )
    print("Đã bổ sung hàm đọc TextGrid.")
else:
    assert "def align_textgrid(" in noi_dung
    print("Module đã có hai hàm, giữ nguyên.")

compile(
    tep_alignment.read_text(encoding="utf-8"),
    str(tep_alignment),
    "exec",
)

print("Module không có lỗi cú pháp.")

Đã bổ sung hàm đọc TextGrid.
Module không có lỗi cú pháp.


In [11]:
%pip install praatio

In [12]:
import json
import subprocess
from pathlib import Path

goc = Path("/content/drive/MyDrive/CMU-MOSI")

thu_muc_ket_qua = (
    goc
    / "preprocessing/alignment/pilot/20261102T093614_520589Z"
    / "alignment_20261102T093905_856138Z"
)

tep_ghep_cu = thu_muc_ket_qua / "text_token_timestamps.json"
thu_muc_textgrids = thu_muc_ket_qua / "textgrids"

assert tep_ghep_cu.is_file(), tep_ghep_cu
assert thu_muc_textgrids.is_dir(), thu_muc_textgrids

# Kiểm tra trong Python hiện tại có praatio chưa
import importlib.util
import sys

if importlib.util.find_spec("praatio") is not None:
    lenh_python = [sys.executable]
else:
    micromamba = Path("/content/mfa_tools/bin/micromamba")
    mfa_env = Path("/content/mfa_env")

    assert micromamba.is_file() and mfa_env.is_dir(), (
        "Phiên mới chưa có praatio. "
        "Chạy cell %pip install praatio rồi chạy lại cell này."
    )

    lenh_python = [
        str(micromamba), "run", "-p", str(mfa_env), "python"
    ]

code_kiem_tra = r"""
import importlib.util
import json
import sys
from pathlib import Path

tep_module = Path(sys.argv[1])
thu_muc = Path(sys.argv[2])
tep_cu = Path(sys.argv[3])

dac_ta = importlib.util.spec_from_file_location(
    "mosi_alignment", tep_module
)
module = importlib.util.module_from_spec(dac_ta)
dac_ta.loader.exec_module(module)

ban_cu = json.loads(tep_cu.read_text(encoding="utf-8"))
assert len(ban_cu) == 10

cac_truong = [
    "word_start", "word_end",
    "token_start", "token_end", "token_time_mask",
]

so_khop = 0

for id_mau, text in ban_cu.items():
    cac_tep = [
        p for p in thu_muc.rglob("*.TextGrid")
        if p.stem == id_mau
    ]

    assert len(cac_tep) == 1, f"TextGrid thiếu hoặc trùng: {id_mau}"

    ket_qua = module.align_textgrid(
        cac_tep[0],
        text["words"],
        text["word_ids"],
    )

    for truong in cac_truong:
        assert ket_qua[truong] == text[truong], (
            f"Khác kết quả: {id_mau}, {truong}"
        )

    # Cố tình sửa một từ: module phải từ chối
    words_sai = list(text["words"])
    words_sai[0] = "__tu_co_tinh_khong_khop__"

    try:
        module.align_textgrid(
            cac_tep[0], words_sai, text["word_ids"]
        )
    except ValueError:
        pass
    else:
        raise AssertionError(f"Không chặn từ sai: {id_mau}")

    so_khop += 1
    print("PASS:", id_mau)

print(f"Đọc TextGrid và ánh xạ khớp: {so_khop}/10")
print("Chặn danh sách từ sai: 10/10")
"""

ket_qua = subprocess.run(
    lenh_python + [
        "-c",
        code_kiem_tra,
        str(tep_alignment),
        str(thu_muc_textgrids),
        str(tep_ghep_cu),
    ],
    capture_output=True,
    text=True,
)

print(ket_qua.stdout)

if ket_qua.returncode != 0:
    print(ket_qua.stderr)

ket_qua.check_returncode()

PASS: 7JsX8y1ysxY$_$12
PASS: 8OtFthrtaJM$_$23
PASS: 8d-gEyoeBzc$_$5
PASS: 9T9Hf74oK10$_$9
PASS: Bfr499ggo-0$_$12
PASS: I5y0__X72p0$_$35
PASS: I5y0__X72p0$_$6
PASS: Nzq88NnDkEk$_$32
PASS: VbQk4H8hgr0$_$24
PASS: Vj1wYRQjB-o$_$2
Đọc TextGrid và ánh xạ khớp: 10/10
Chặn danh sách từ sai: 10/10



In [13]:
tep_huong_dan = goc / "src/preprocessing/README_alignment.md"

tep_huong_dan.write_text(
    """# Module alignment CMU-MOSI

## Phụ thuộc
pip install praatio

## Giao diện
- read_textgrid(textgrid_path): đọc tier words, bỏ khoảng im lặng.
- process_alignment(word_intervals, words, word_ids, duration=None):
  kiểm tra khoảng từ đã đọc và ánh xạ timestamp sang token.
- align_textgrid(textgrid_path, words, word_ids, duration=None):
  đọc TextGrid và gọi process_alignment.

## Đầu vào
- words và word_ids lấy từ process_text.
- duration là thời lượng WAV tính bằng giây, nếu có.

## Đầu ra
words, word_ids, word_intervals,
word_sequence_match, timestamp_structure_pass,
word_start, word_end, token_start, token_end, token_time_mask.

## Quy tắc
- Chỉ bỏ qua khác biệt hoa/thường khi đối chiếu từ.
- Từ thiếu, khác hoặc tách khác phải báo ValueError.
- Không ghép timestamp theo thứ tự khi danh sách từ không khớp.
- Token đặc biệt word_id=-1 nhận thời gian None và mask False.
- Các subword cùng từ nhận cùng khoảng thời gian của từ.
- Kiểm tra thời gian hữu hạn, dương và không chồng lấn.
- Kiểm tra giới hạn thời lượng khi truyền duration.
- Không tự sửa transcript hoặc trạng thái nghe kiểm tra.

## Kiểm chứng
Đã đối chiếu module với kết quả notebook trên 10 mẫu pilot.
Chưa chạy alignment toàn bộ 2.199 mẫu.

## Kết quả Text Full trên Drive
preprocessing/text/full/20261102T152904_636429Z/
Text và BERT embedding: 2.199 mẫu thành công.
Embedding hiện theo token, tầng cuối BERT; chưa gộp subword.
Cấu hình đầu vào DeepCU cần được nhóm thống nhất.
""",
    encoding="utf-8",
)

print("Đã lưu:", tep_huong_dan)

Đã lưu: /content/drive/MyDrive/CMU-MOSI/src/preprocessing/README_alignment.md
